# Моніторинг та Алертинг: Logistics SLA

Цей ноутбук забезпечує безперервний моніторинг ефективності логістики. Згідно з бізнес-вимогами, ми відстежуємо частку запізнень (delay rate) у часі. Якщо відсоток запізнень (отримання товару після обіцяної дати) перевищує встановлений поріг (15%), скрипт генерує системний алерт. 

У виробничому середовищі Databricks цей код запускається як Job, і у випадку алерту автоматично надсилається сповіщення команді Procurement або Logistics.

In [0]:
import os
import sys
from pathlib import Path

os.environ["DB_CATALOG"] = "workspace"
os.environ["SCHEMA_PREFIX"] = "tpch"

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import env
from src.monitoring.delay_alerts import run_delay_monitoring

print(f"Запуск пайплайну моніторингу на базі {env.CATALOG}.{env.SILVER_SCHEMA}...")
monitoring_results = run_delay_monitoring(spark, threshold_pct=15.0)

metrics_df = monitoring_results["metrics"]
alerts_df = monitoring_results["alerts"]
print("Метрики розраховано успішно.")

## 1. Помісячний тренд затримок (Delay Rate Over Time)
Ця таблиця показує динаміку ефективності доставки. 

In [0]:
display(metrics_df)

## 2. Перевірка тригерів (Alert Rules)
Тут відфільтровані лише ті періоди, де відсоток запізнень перевищив допустиму межу в 15%[cite: 6]. Наступна клітинка з кодом перевіряє наявність таких записів і, якщо вони є, штучно викликає помилку для зупинки пайплайну та активації сповіщень.

In [0]:
display(alerts_df)

alert_count = alerts_df.count()

if alert_count > 0:
    problematic_months = [
        row["receipt_month"].strftime("%Y-%m") for row in alerts_df.collect()
    ]
    
    raise AssertionError(
        f"🚨 АЛЕРТ: Виявлено {alert_count} місяців із рівнем затримок > 15%! "
        f"Проблемні періоди: {', '.join(problematic_months)}. "
        "Необхідна увага менеджера з логістики."
    )
else:
    print("✅ Усі показники в нормі. Рівень затримок залишається в межах допустимого SLA (<= 15%).")